This notebook is for evaluating LexGlue. We are going to evaluate all of LexGlUE's datasets, including:


1.   SCOTUS
2.   ECtHR A
3.   ECtHR B
4.   LEDGAR
5.   EURLEX
6.   UNFAIR-ToS
7.   CaseHOLD

We are going to examine the accuracy (μ-F1 and m-F1) of RLM with this benchmark, and compare it with vanilla LLM.



In [23]:
# package instalizations

%pip install rlms
%pip install datasets
%pip install scikit-learn


[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 24.3.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [24]:
# imports

import os
from rlm import RLM
from datasets import load_dataset
from collections.abc import Sequence

In [25]:
# Different LLMs API Keys

os.environ['OPENROUTER_API_KEY'] = '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'
os.environ['GEMINI_API_KEY'] = 'AQ.Ab8RN6L0opJavYC3imR-sg2dXBr8YK8rSVFjDSXz-cw3f97i7Q'
os.environ['GROQ_API_KEY'] = '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'

In [64]:
# RLM Object instantiation

rlm = RLM(
    backend="gemini",
    backend_kwargs={"model_name": "gemini-2.5-flash"},
    verbose=True,  # Keep this True! It lets you watch the RLM write and execute Python code in real-time.
)

╭─ ◆ RLM ━ Recursive Language Model ──────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│    Backend                         gemini                          Environment                     local        │
│    Model                           gemini-2.5-flash                Max Iterations                  30           │
│    Max Depth                       1                                                                            │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

We define a function for prompt generation, based on the **Task Type** of these datasets. it can be:


*   multi-**class**-classification : 1 document → exactly 1 label
*   multi-**label**-classification : 1 document → 1 or more labels
*   multiple-choice QA : 1 document + n possible holdings → only 1 holding is correct



In [27]:
TASK_TYPE: dict[int] = {
    "MULTI_CLASS": 1,
    "MULTI_LABEL": 2,
    "MULTIPLE_CHOICE_QA": 3
}


In [28]:
# extract datasets labels based on task type
def extract_labels(task_type: int, dataset_features) -> Sequence[str]:
  if task_type == TASK_TYPE["MULTI_CLASS"]:
    return dataset_features['label'].names

  if task_type == TASK_TYPE["MULTI_LABEL"]:
    return dataset_features['labels'].feature.names

  else:
    raise KeyError(f"The following task type is invalid ({task_type}). It should be among this list: {TASK_TYPE.values()}")


In [29]:
# 5. EURLEX

dataset_eurlex = load_dataset("coastalcph/lex_glue", "eurlex")
dataset_label_ids = extract_labels(2, dataset_eurlex['validation'].features)
print(extract_labels(2, dataset_eurlex['validation'].features))

['100163', '100168', '100169', '100170', '100171', '100172', '100173', '100174', '100175', '100176', '100177', '100179', '100180', '100183', '100184', '100185', '100186', '100187', '100189', '100190', '100191', '100192', '100193', '100194', '100195', '100196', '100197', '100198', '100199', '100200', '100201', '100202', '100204', '100205', '100206', '100207', '100212', '100214', '100215', '100220', '100221', '100222', '100223', '100224', '100226', '100227', '100229', '100230', '100231', '100232', '100233', '100234', '100235', '100237', '100238', '100239', '100240', '100241', '100242', '100243', '100244', '100245', '100246', '100247', '100248', '100249', '100250', '100252', '100253', '100254', '100255', '100256', '100257', '100258', '100259', '100260', '100261', '100262', '100263', '100264', '100265', '100266', '100268', '100269', '100270', '100271', '100272', '100273', '100274', '100275', '100276', '100277', '100278', '100279', '100280', '100281', '100282', '100283', '100284', '100285']

In [30]:
# for EURLEX, we need to get and build the dictionary and list of EuroVoc

import requests

EUROVOC_URL = (
    "https://raw.githubusercontent.com/"
    "nlpaueb/multi-eurlex/master/data/eurovoc_descriptors.json"
)

# Download and return the EuroVoc descriptors.
def load_eurovoc() -> dict:
    response = requests.get(EUROVOC_URL)
    response.raise_for_status()
    return response.json()


# return the mapping between vocab code and english equivalent
def get_eurovoc_label_map(dataset_label_ids) -> dict[str, str]:
    data = load_eurovoc()
    return {
        label_id: data[label_id]["en"]
        for label_id in dataset_label_ids
    }


# return the list of EuroVoc list (i.e EuroLex label names)
def get_eurovoc_label_names(dataset_label_ids) -> list[str]:
    label_map = get_eurovoc_label_map(dataset_label_ids)
    return [label_map[label_id] for label_id in dataset_label_ids]

eurolex_label_names = get_eurovoc_label_names(dataset_label_ids)
print(len(eurolex_label_names))

100


In [31]:
# label names are imported from the offocial github repo for evaluating
# of one-shot test of lexGlue + llm (https://github.com/coastalcph/zeroshot_lexglue.git)

LEXGLUE_TASKS_DICT: dict[dict[str]] = {
    "SCOTUS": {
        "task_type": 1,
        "task_description": "Classify the United States Supreme Court opinion according to its primary legal issue area.",
        "label_names": ['Criminal Procedure', 'Civil Rights', 'First Amendment', 'Due Process', 'Privacy', 'Attorneys',
                'Unions', 'Economic Activity', 'Judicial Power', 'Federalism', 'Interstate Relations',
                'Federal Taxation', 'Miscellaneous']
    },
    "ECtHR_A": {
        "task_type": 2,
        "task_description": "Predict which Articles of the European Convention on Human Rights the applicant alleges were violated.",
        "label_names": ['Article 2 - Right to life', 'Article 3 - Prohibition of torture',  'Article 5 - Right to liberty and security',
                        'Article 6 - Right to a fair trial',  'Article 8 - Right to respect for private and family life',
                        'Article 9 - Freedom of thought, conscience and religion', 'Article 10 - Freedom of expression',
                        'Article 11 - Freedom of assembly and association', 'Article 14 - Prohibition of discriminatioN',
                        'Article P1-1 - Protection of property']
    },
    "ECtHR_B": {
        "task_type": 2,
        "task_description": "Predict which Articles of the European Convention on Human Rights the Court found to have been violated.",
        "label_names": ['Article 2 - Right to life', 'Article 3 - Prohibition of torture',  'Article 5 - Right to liberty and security',
                        'Article 6 - Right to a fair trial',  'Article 8 - Right to respect for private and family life',
                        'Article 9 - Freedom of thought, conscience and religion', 'Article 10 - Freedom of expression',
                        'Article 11 - Freedom of assembly and association', 'Article 14 - Prohibition of discriminatioN',
                        'Article P1-1 - Protection of property']
    },
    "LEDGAR": {
        "task_type": 1,
        "task_description": "Classify the contract provision according to its clause category.",
        "label_names": None
    },
    "EURLEX": {
        "task_type": 2,
        "task_description": "Assign all applicable EuroVoc legal concepts to this European Union legal document.",
        "label_names": eurolex_label_names
    },
    "UNFAIR_ToS": {
        "task_type": 2,
        "task_description": "Identify all categories of potentially unfair contractual terms that apply to this Terms of Service sentence.",
        "label_names": None
    },
    "CaseHOLD": {
        "task_type": 3,
        "task_description": "",
        "label_names": None
    }
}

In [32]:
def get_data_item_label_attr_name(task_type: int) -> str:
  if task_type == TASK_TYPE["MULTI_CLASS"] or task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]:
    return 'label'

  return 'labels'

In [33]:
# Builds a mapping from dataset label -> human-readable label.
# only for the datasets with not-None value as label_name in LEXGLUE_TASKS_DICT

def build_label_map(
    dataset_labels: Sequence[str],
    readable_labels: Sequence[str],
) -> dict[str, str]:

    print(dataset_labels)
    print(readable_labels)


    if len(dataset_labels) != len(readable_labels):
        raise ValueError(
            f"Length mismatch: {len(dataset_labels)} != {len(readable_labels)}"
        )

    return dict(zip(dataset_labels, readable_labels))

In [34]:
def multi_class_classification_prompt(task_name: str, dataset_features, data_item) -> str:

  task_description = LEXGLUE_TASKS_DICT[task_name]["task_description"]
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    labels = extract_labels(task_type=TASK_TYPE["MULTI_CLASS"], dataset_features=dataset_features)
  else:
    labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]

  return f"""
You are performing a multi-class legal document classification task.

Task: {task_description}

Choose exactly ONE label from the candidate labels below.

Candidate labels: {labels}

Output requirements:
- Return only the exact label name from the candidate labels.
- Do not add explanations, punctuation, or additional text.

Document:
<document>
{data_item['text']}
</document>

""".strip()

In [35]:
def multi_label_classification_prompt(task_name: str, dataset_features, data_item) -> list[str]:

  task_description = LEXGLUE_TASKS_DICT[task_name]["task_description"]
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    labels = extract_labels(task_type=TASK_TYPE["MULTI_LABEL"], dataset_features=dataset_features)
  else:
    labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]

  return f"""
You are performing a multi-label legal document classification task.

Task: {task_description}

Select every applicable label from the candidate labels below.
Zero, one, or multiple labels may apply.

Candidate labels: {labels}

Output requirements:
- Return the applicable labels as a python list from the candidate labels.
- Do not add explanations or additional text.
- If no label applies, return NONE.

Document:
<document>
{data_item['text']}
</document>

""".strip()

In [36]:
def multiple_choice_QA_prompt(data_item) -> str:
  holdings = data_item['endings']

  return f"""
  You are given an excerpt from a judicial decision and five candidate holdings.

  Select the candidate holding that correctly completes or corresponds to the
  legal reasoning in the provided context.

  Return only the index of the corresponding holding.
  Do not provide an explanation.

  Legal context:
  <context>
  {data_item['context']}
  </context>

  Candidate holdings:
  {holdings}
  """.strip()


In [37]:

def generate_prompt(task_name: str, data_features, data_item: str) -> str:

  if task_name not in LEXGLUE_TASKS_DICT:
    raise KeyError(f'Task name is invalid ({task_name}). supported tasks: {", ".join(LEXGLUE_TASKS_DICT)}')

  task_type = LEXGLUE_TASKS_DICT[task_name]['task_type']
  if task_type not in TASK_TYPE.values():
    raise KeyError(f'Task type is invalid ({task_type}). supported task types: {", ".join(TASK_TYPE)}')

  # Case 1: multi-class-classification
  if task_type == TASK_TYPE['MULTI_CLASS']:
    return multi_class_classification_prompt(task_name, data_features, data_item)

  # Case 2: multi-label-classification
  if task_type == TASK_TYPE["MULTI_LABEL"]:
    return multi_label_classification_prompt(task_name, data_features, data_item)
    # Case 3: multiple-choice QA
  return multiple_choice_QA_prompt(data_item)




- We are now working with the validation set, and then, work with the test set.

**--------------------**

**Multi Class Tasks:**

**--------------------**

In [38]:
# 1. SCOTUS (US Supreme Court)

dataset_scotus = load_dataset("coastalcph/lex_glue", "scotus")
print(extract_labels(1, dataset_scotus['validation'].features))
print(len(dataset_scotus['validation']))

['1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13']
1400


In [39]:
# 2. LEDGAR

dataset_ledgar = load_dataset("coastalcph/lex_glue", "ledgar")
print(extract_labels(1, dataset_ledgar['validation'].features))
print(len(dataset_ledgar['validation']))

['Adjustments', 'Agreements', 'Amendments', 'Anti-Corruption Laws', 'Applicable Laws', 'Approvals', 'Arbitration', 'Assignments', 'Assigns', 'Authority', 'Authorizations', 'Base Salary', 'Benefits', 'Binding Effects', 'Books', 'Brokers', 'Capitalization', 'Change In Control', 'Closings', 'Compliance With Laws', 'Confidentiality', 'Consent To Jurisdiction', 'Consents', 'Construction', 'Cooperation', 'Costs', 'Counterparts', 'Death', 'Defined Terms', 'Definitions', 'Disability', 'Disclosures', 'Duties', 'Effective Dates', 'Effectiveness', 'Employment', 'Enforceability', 'Enforcements', 'Entire Agreements', 'Erisa', 'Existence', 'Expenses', 'Fees', 'Financial Statements', 'Forfeitures', 'Further Assurances', 'General', 'Governing Laws', 'Headings', 'Indemnifications', 'Indemnity', 'Insurances', 'Integration', 'Intellectual Property', 'Interests', 'Interpretations', 'Jurisdictions', 'Liens', 'Litigations', 'Miscellaneous', 'Modifications', 'No Conflicts', 'No Defaults', 'No Waivers', 'Non-

**--------------------**

**Multi Label Tasks:**

**--------------------**

In [66]:
# 3. ECtHR A

dataset_ecthr_a = load_dataset("coastalcph/lex_glue", "ecthr_a")
print(extract_labels(2, dataset_ecthr_a['validation'].features))
print(len(dataset_ecthr_a['validation']))


['2', '3', '5', '6', '8', '9', '10', '11', '14', 'P1-1']
1000


In [41]:
# Also EURLEX belongs here, but since we needed to get the EuroVoc labels, we have already done it above.

In [42]:
# 4. ECtHR B

dataset_ecthr_b = load_dataset("coastalcph/lex_glue", "ecthr_b")
print(extract_labels(2, dataset_ecthr_b['validation'].features))
print(len(dataset_ecthr_b['validation']))

['2', '3', '5', '6', '8', '9', '10', '11', '14', 'P1-1']
1000


In [43]:
# 6. UNFAIR-ToS

dataset_unfair_tos = load_dataset("coastalcph/lex_glue", "unfair_tos")
print(extract_labels(2, dataset_unfair_tos['validation'].features))
print(len(dataset_unfair_tos['validation']))

['Limitation of liability', 'Unilateral termination', 'Unilateral change', 'Content removal', 'Contract by using', 'Choice of law', 'Jurisdiction', 'Arbitration']
2275


In [44]:
# 7. Case Hold

dataset_casehold = load_dataset("coastalcph/lex_glue", "case_hold")
print(dataset_casehold['validation'][0])
print(len(dataset_casehold['validation']))

{'context': 'Supply, plaintiff in its complaint and in its motion for summary judgment before the bankruptcy court vigorously contended that the financing statements comported with the statutory requirements and constituted the “first and best lien” on EAI of NC’s assets. Plaintiff asserted that the statutory and case law established that: {¶ 15} “[I]nadvertent omission [in plaintiffs financing statements] is not seriously misleading and does not render the Financing Statements invalid because: a) the mere omission of the words ‘of North Carolina’ when the addresses of EAI and EAI of NC are the same and the two are related companies does not result in a misleading error; and b) the coversheets specifically refer the reader to the attached Security Agreement for the signature 3 UCC Rep.Serv. 719 (<HOLDING>). {¶ 18} Plaintiff prevailed and the bankruptcy', 'endings': ['holding a forged signature constitutes the use of that persons name and thus qualifies as a means of identification unde

Here, we choose one of the datasets above:

In [45]:
def get_label_key_ids(task_name: str, label_index: int|list[int], dataset_features) -> str | list[str]:
  dataset_labels = extract_labels(LEXGLUE_TASKS_DICT[task_name]["task_type"], dataset_features)
  if (type(label_index) == list):
    result = []
    for l in label_index:
      result.append(dataset_labels[l])
    return result

  return dataset_labels[label_index]


def get_actual_readable_label_name(task_name: str, label_index: int|list[int], dataset_features) -> str | list[str]:
  actual_label = get_label_key_ids(task_name, label_index, dataset_features)
  if (LEXGLUE_TASKS_DICT[task_name]["label_names"] == None):
    return actual_label
  else:
    readable_labels = LEXGLUE_TASKS_DICT[task_name]["label_names"]
    dataset_labels = extract_labels(LEXGLUE_TASKS_DICT[task_name]["task_type"], dataset_features)
    label_dict = build_label_map(dataset_labels, readable_labels)
    if (type(actual_label) == list):
      result = []
      for l in actual_label:
        result.append(label_dict[l])
      return result

    return label_dict[actual_label]

Adding a caching mechanism functions for caching repetitive llm calls:

In [59]:
from pathlib import Path

CACHE_DIR = Path("./thesis_rlm_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

In [120]:
import json
import hashlib
from pathlib import Path
from typing import Any

CACHE_DIR = Path("./thesis_cache")
CACHE_DIR.mkdir(parents=True, exist_ok=True)


def get_cache_file(
    task_name: str,
    split_name: str,
    method_name: str,
    model_name: str,
    temperature: float,
) -> Path:
    safe_model_name = model_name.replace("/", "_")
    filename = (
        f"{task_name}-{split_name}-{method_name}-"
        f"{safe_model_name}-temp{temperature}.json"
    )
    print(f"Cache file path: {CACHE_DIR / filename}")
    return CACHE_DIR / filename


def load_cache(cache_file: Path) -> dict[str, Any]:
    if not cache_file.exists():
        return {}

    try:
        with cache_file.open("r", encoding="utf-8") as file:
            return json.load(file)
    except (json.JSONDecodeError, OSError):
        return {}


def save_cache(cache_file: Path, cache: dict[str, Any]) -> None:
    temp_file = cache_file.with_suffix(".tmp")

    with temp_file.open("w", encoding="utf-8") as file:
        json.dump(cache, file, ensure_ascii=False, indent=2)

    temp_file.replace(cache_file)


def get_task_text_hash(prompt: str) -> str:
    return hashlib.sha256(prompt.encode("utf-8")).hexdigest()[:16]

The configuration for the current dataset in evaluation:

In [194]:
# Configuration

split_name = "validation"
method_name = "rlm"
model_name = "gemini-2.5-flash"
current_dataset = dataset_casehold
current_dataset_features = current_dataset[split_name].features
current_task_name = "CaseHOLD"
max_sample = 100

In [206]:
# For fixing the missed samples

task_type = LEXGLUE_TASKS_DICT[current_task_name]['task_type']
label_attr = get_data_item_label_attr_name(task_type=task_type)
cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)
index =  71
data_item = current_dataset[split_name][index]  # Example data item for demonstration

prompt = generate_prompt(
task_name=current_task_name,
data_features=current_dataset_features,
data_item=data_item
)

cache_key = str(index)
text_hash = get_task_text_hash(prompt)

cached_item = cache.get(cache_key)


result = rlm.completion(prompt=prompt)
prediction = result.response

label_index = data_item[label_attr]
if (task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]):
    actual_res = label_index
else:
    actual_res = get_actual_readable_label_name(
    task_name=current_task_name,
    label_index=label_index,
    dataset_features=current_dataset_features
)

cache[cache_key] = {
        "prediction": prediction,
        "actual_label": actual_res,
        "text_hash": text_hash,
}

save_cache(cache_file, cache)


Cache file path: thesis_cache/CaseHOLD-validation-rlm-gemini-2.5-flash-temp0.json


──────────────────────────────────────────────────  Iteration 1  ──────────────────────────────────────────────────

╭─ ◇ LLM Response  (1.37s) ───────────────────────────────────────────────────────────────────────────────────────╮
│ ```repl                                                                                                         │
│ print(context)                                                                                                  │
│ print(f"Context length: {len(context)} characters.")                                                            │
│ ```                                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│ ~7 words                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ▸ Code Execution  (0.001s) ────────────────────────────────────────────────────────────────────────────────────╮
│ Code:                                                                                                           │
│ print(context)                                                                                                  │
│ print(f"Context length: {len(context)} characters.")                                                            │
│                                                                                                                 │
│ Output:                                                                                                         │
│ You are given an excerpt from a judicial decision and five candidate holdings.                                  │
│                                                                                                                 │
│   Select the candidate holding that correctly completes or corresponds to the                                   │
│   legal reasoning in the provided context.                                                                      │
│                                                                                                                 │
│   Return only the index of the corresponding holding.                                                           │
│   Do not provide an explanation.                                                                                │
│                                                                                                                 │
│   Legal context:                                                                                                │
│   <context>                                                                                                     │
│   mere anxiety or annoyance. Because of our ruling in Lopez, which was well argued before the trial court in    │
│ this case, I would hold that the court erred in its application of the emotional distress prong of the stalking │
│ statute to the findings in this case. 127 In reviewing the trial court's application of the second element of   │
│ the stalking statute to the facts in this case, the findings support the trial court's conclusion that Bayles   │
│ knew or should have known that his conduct would cause emotional distress to Bailey. However, the court did not │
│ specifically find, in support of this element, that plaintiff suffered from more than anxiety and annoyance,    │
│ perhaps because plaintiff put on no objective evidence thereof. CJ Hansen v. Mountain Fuel Supply Co., 858 P.2d │
│ 970, 973-75 (Utah 1993) (<HOLDING>); White v. Blackburn, 787 P.2d 1315, 1317-18                                 │
│   </context>                                                                                                    │
│                                                                                                                 │
│   Candidate holdings:                                                                                           │
│   ['holding that emotional distress requires a showing of either physical symptoms or mental illness', 'holding │
│ that employee could not state a claim for negligent infliction of emotional distress under fela without a       │
│ showing of physical impact', 'holding that title 42 usc  1997ee requires proof of a physical injury before a    │
│ prisoner can recover for emotional or mental distress and that verbal abuse is not actionable under section     │
│ 1983', 'holding in a fcra case that plaintiffs may not rely on mere  conclusory statements  rather they must    │
│ sufficiently articulate  true demonstrable emotional distress  including the factual context in which the       │
│ emotional distress arose evidence corroborating the testimony of the plaintiff the nexus between the conduct of │
│ the defendant and the emotional distress the degree of

──────────────────────────────────────────────────  Iteration 2  ──────────────────────────────────────────────────

...

╭─   ↳ LLM Sub-call: DEFAULT_MODEL  (5.32s) ──────────────────────────────────────────────────────────────────────╮
│ Prompt:                                                                                                         │
│ You are given an excerpt from a judicial decision and five candidate holdings.                                  │
│                                                                                                                 │
│ Select the candidate holding that correctly completes or corresponds to the                                     │
│ legal reasoning in the provided context.                                                                        │
│                                                                                                                 │
│ R…                                                                                                              │
│ Response: 1                                                                                                     │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  1                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       3                                                   
                                              Total Time       12.93s                                              
                                              Input Tokens     6,720                                               
                                              Output Tokens    1,039   

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

In [127]:
task_type = LEXGLUE_TASKS_DICT[current_task_name]['task_type']
label_attr = get_data_item_label_attr_name(task_type=task_type)
cache_file = get_cache_file(
    task_name=current_task_name,
    split_name=split_name,
    method_name=method_name,
    model_name=model_name,
    temperature=0,
)

cache = load_cache(cache_file)

for index, data_item in enumerate(current_dataset[split_name]):
  if max_sample > 0 and index >= max_sample:
    break
  prompt = generate_prompt(
    task_name=current_task_name,
    data_features=current_dataset_features,
    data_item=data_item
  )

  cache_key = str(index)
  text_hash = get_task_text_hash(prompt)

  cached_item = cache.get(cache_key)
  
  if (
        cached_item is not None
        and cached_item.get("text_hash") == text_hash
    ):
      prediction = cached_item["prediction"]
      actual_res = cached_item["actual_label"]
      source = "cache"
      print(f"Cache hit for sample index {index}. Using cached result.")

  else:
    result = rlm.completion(prompt=prompt)
    prediction = result.response

    label_index = data_item[label_attr]
    if (task_type == TASK_TYPE["MULTIPLE_CHOICE_QA"]):
      actual_res = label_index
    else:
      actual_res = get_actual_readable_label_name(
        task_name=current_task_name,
        label_index=label_index,
        dataset_features=current_dataset_features
      )
    
    cache[cache_key] = {
          "prediction": prediction,
          "actual_label": actual_res,
          "text_hash": text_hash,
    }

    save_cache(cache_file, cache)
    source = "API"
    print(f"Cache miss for sample index {index}. Using API result and saving to cache.")

  print(f"data row number {index + 1}.--------------------")
  print(f"Predicted: {prediction}, actual: {actual_res}")
  print("---------------------------")


Cache file path: thesis_cache/CaseHOLD-validation-rlm-gemini-2.5-flash-temp0.json
Cache hit for sample index 0. Using cached result.
data row number 1.--------------------
Predicted: 3, actual: 3
---------------------------
Cache hit for sample index 1. Using cached result.
data row number 2.--------------------
Predicted: 1, actual: 0
---------------------------
Cache hit for sample index 2. Using cached result.
data row number 3.--------------------
Predicted: 4, actual: 4
---------------------------
Cache hit for sample index 3. Using cached result.
data row number 4.--------------------
Predicted: 4, actual: 3
---------------------------
Cache hit for sample index 4. Using cached result.
data row number 5.--------------------
Predicted: ERROR: The `llm_query` function (and other LLM interaction functions) described as available were not found in the REPL environment. `SHOW_VARS()` confirmed their absence. Therefore, I cannot complete the task as it requires semantic understanding o

──────────────────────────────────────────────────  Iteration 1  ──────────────────────────────────────────────────

╭─ ◇ LLM Response  (1.22s) ───────────────────────────────────────────────────────────────────────────────────────╮
│ ```repl                                                                                                         │
│ print(context)                                                                                                  │
│ ```                                                                                                             │
│                                                                                                                 │
│                                                                                                                 │
│ ~3 words                                                                                                        │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭─ ▸ Code Execution  (0.001s) ────────────────────────────────────────────────────────────────────────────────────╮
│ Code:                                                                                                           │
│ print(context)                                                                                                  │
│                                                                                                                 │
│ Output:                                                                                                         │
│ You are given an excerpt from a judicial decision and five candidate holdings.                                  │
│                                                                                                                 │
│   Select the candidate holding that correctly completes or corresponds to the                                   │
│   legal reasoning in the provided context.                                                                      │
│                                                                                                                 │
│   Return only the index of the corresponding holding.                                                           │
│   Do not provide an explanation.                                                                                │
│                                                                                                                 │
│   Legal context:                                                                                                │
│   <context>                                                                                                     │
│   were not alleged in the indictment, whether he was an organizer or leader should not have been submitted to   │
│ the jury. Because the jury concluded beyond a reasonable doubt that Soreide was an organizer or leader, and     │
│ because Soreide was sentenced to 236 months imprisonment, which was below the statutory maximum of 20 years     │
│ imprisonment, the district court did not unconstitutionally amend the indictment by enhancing Soreide’s         │
│ sentence based on his role as an organizer or leader. As to Soreide’s argument that the district court failed   │
│ to make the necessary findings under 18 U.S.C. § 3553(a), we have held that a sentencing court is not obligated │
│ to specifically address and analyze on the record every factor set forth in § 3553(a). United States v. Scott,  │
│ 426 F.3d 1324, 1329 (11th Cir.2005) (<HOLDING>). We find that the district court adequately                     │
│   </context>                                                                                                    │
│                                                                                                                 │
│   Candidate holdings:                                                                                           │
│   ['holding that a district court does not have to state on the record that it considered every  3553a factor', │
│ 'recognizing that many of the  3553a factors are already incorporated into any guidelines determination and the │
│ 3553a factors can themselves overlap', 'holding that nothing in booker or elsewhere requires the district court │
│ to state on the record that it has explicitly considered each of the  3553a factors or to discuss each of the   │
│ 3553a factors', 'holding that to establish reasonableness of a sentence a district court need not explicitly    │
│ discuss every  3553a factor on the record', 'holding a sentencing court is not obligated to expressly weigh on  │
│ the record each of the factors set out in  3553a']                                                              │
│                                                                                                                 │
╰───────────────────────────────────────────────────────

──────────────────────────────────────────────────  Iteration 2  ──────────────────────────────────────────────────

...

╭─ ★ Final Answer ────────────────────────────────────────────────────────────────────────────────────────────────╮
│                                                                                                                 │
│  3                                                                                                              │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

  Iterations       3                                                   
                                              Total Time       12.33s                                              
                                              Input Tokens     4,974                                               
                                              Output Tokens    528     

═══════════════════════════════════════════════════════════════════════════════════════════════════════════════════

Cache miss for sample index 99. Using API result and saving to cache.
data row number 100.--------------------
Predicted: 3, actual: 2
---------------------------
